# GC-SUAE: Reproduction Notebook
**Geologically-Constrained Spectral Unmixing Autoencoder for Unsupervised Lunar Mineral Mapping**

This notebook reproduces all results in the paper on Google Colab (T4 GPU).

**Requirements before running:**
- Google Drive with the three data files mounted (IIRS .hdr+.qub, TMC-2 DEM .tif, Clementine FeO .tif)
- Colab runtime set to GPU (Runtime → Change runtime type → T4 GPU)

## 1. Environment Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Set your data paths here 
BASE_DIR = '/content/drive/MyDrive/Lunar_dataset'
IIRS_HDR = f'{BASE_DIR}/data.hdr'
IIRS_QUB = f'{BASE_DIR}/data.qub'
TMC_DEM  = f'{BASE_DIR}/ch2_tmc_ndn_20200703T1535027868_d_dtm_d18.tif'
FEO_MAP  = f'{BASE_DIR}/Lunar_Clementine_UVVIS_FeO_ClrBinned_70S70N_1km.tif'

import os
for path in [IIRS_HDR, IIRS_QUB, TMC_DEM, FEO_MAP]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    print(f"{'OK' if exists else 'MISSING'} | {size:7.1f} MB | {os.path.basename(path)}")

In [ ]:
%%capture
!pip install spectral rasterio scipy scikit-learn matplotlib seaborn pyyaml tqdm pytest torch torchvision

In [ ]:
import os
if not os.path.exists('/content/GC-SUAE'):
    !git clone https://github.com/Vaibhavtripathi7/GC-SUAE.git /content/GC-SUAE

os.chdir('/content/GC-SUAE')
!pip install -e . --quiet
print('Repo ready:', os.getcwd())

## 2. Verify Environment

In [ ]:
!python -m pytest tests/test_models.py -v
# Expected: 19 passed

## 3. Prepare Endmember Library

In [ ]:
!python scripts/prepare_endmembers.py
# Creates data/endmembers/relab_lunar_6minerals.npy
# 6 synthetic spectra: plagioclase, orthopyroxene, clinopyroxene,
#                      olivine, ilmenite, spinel (86 bands, 800-2500 nm)

## 4. Preprocess and Align Modalities

In [ ]:
!python scripts/preprocess_data.py \
    --iirs_hdr "{IIRS_HDR}" \
    --iirs_qub "{IIRS_QUB}" \
    --tmc_dem  "{TMC_DEM}" \
    --feo_map  "{FEO_MAP}" \
    --output_dir data/processed/

# Expected output:
# [Preprocess] IIRS shape: 5622x250x256 bands
# [Preprocess] DEM aligned to IIRS grid: (5622, 250)
# [Preprocess] FeO aligned to IIRS grid: (5622, 250)
# [Preprocess] Saved: data/processed/aligned_tmc2_dem.tif
# [Preprocess] Saved: data/processed/aligned_elemental_map.tif

## 5. Configure and Run Ablation

In [ ]:
import yaml

with open('configs/ablation.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data']['iirs_hdr']       = IIRS_HDR
cfg['data']['iirs_qub']       = IIRS_QUB
cfg['data']['dem_path']       = 'data/processed/aligned_tmc2_dem.tif'
cfg['data']['feo_path']       = 'data/processed/aligned_elemental_map.tif'
cfg['data']['endmember_path'] = 'data/endmembers/relab_lunar_6minerals.npy'

with open('configs/ablation.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('Config updated')
print(f'  FeO threshold:   {cfg["data"]["feo_positive_threshold"]} wt%')
print(f'  Slope threshold: {cfg["data"]["slope_positive_threshold"]}')
print(f'  Patch size:      {cfg["data"]["patch_size"]}x{cfg["data"]["patch_size"]}')
print(f'  Stride:          {cfg["data"]["stride"]}')
print(f'  Num clusters:    {cfg["evaluation"]["clustering"]["n_clusters"]}')

In [ ]:
# Create results directory on Drive for persistence
RESULTS_DIR = '/content/drive/MyDrive/LunarSpecNet_results'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)

# Run full ablation 
!python scripts/run_ablation.py \
    --config configs/ablation.yaml \
    2>&1 | tee {RESULTS_DIR}/ablation_log.txt

## 6. Results

In [ ]:
import json

with open('outputs/ablation/ablation_results.json') as f:
    results = json.load(f)

print(f"{'Method':<26} {'Silhouette':>12} {'DB Index':>10} {'SAM (°)':>9} {'Mineral ID':>11}")
print('─' * 72)
for model, m in sorted(results.items()):
    sil  = m.get('silhouette_score', 0)
    db   = m.get('davies_bouldin_index', 0)
    sam  = m.get('mean_sam_deg', 0)
    mid  = m.get('mineral_id_accuracy', 0)
    flag = ' ◄ proposed' if ('NoTAGCL' in model or model == 'GC_SUAE_NoTAGCL') else ''
    print(f"{model:<26} {sil:>12.4f} {db:>10.4f} {sam:>9.3f} {mid*10:>10.0f}/10{flag}")

## 7. Save All Results to Drive

In [ ]:
import shutil, os

# Save results JSON
shutil.copy('outputs/ablation/ablation_results.json',
            f'{RESULTS_DIR}/ablation_results.json')

# Save all model checkpoints
for item in os.listdir('outputs/ablation'):
    src = f'outputs/ablation/{item}'
    dst = f'{RESULTS_DIR}/{item}'
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print(f'Saved: {item}')

print(f'\nAll results saved to {RESULTS_DIR}')